In [ ]:
# ===============================
# CELL 1: IMPORTS & SETUP
# ===============================

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, log_loss,
    roc_curve, accuracy_score
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
import os
import pickle
import glob
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("✅ All imports loaded")


# ===============================
# CELL 2: CONFIG
# ===============================

class DetectorConfig:

    MODELS = {
        'BERT':    'bert-base-uncased',
        'RoBERTa': 'roberta-base',
        'ELECTRA': 'google/electra-base-discriminator',
    }

    HC3_TRAIN  = "hc3_train.csv"
    HC3_TEST   = "hc3_test.csv"
    ELI5_TRAIN = "eli5_train.csv"
    ELI5_TEST  = "eli5_test.csv"

    NUM_EPOCHS       = 1
    BATCH_SIZE_TRAIN = 32
    BATCH_SIZE_EVAL  = 64
    LEARNING_RATE    = 2e-5
    WARMUP_RATIO     = 0.06
    WEIGHT_DECAY     = 0.01
    DROPOUT          = 0.2
    VAL_SPLIT_RATIO  = 0.1
    LOGGING_STEPS    = 100
    MAX_SEQ_LENGTH   = 512
    FP16             = True
    FORCE_RETRAIN    = False

    OUTPUT_DIR_BASE = "./models"
    RESULTS_DIR     = "./results"

    @classmethod
    def get_output_dir(cls, model_name, dataset_name):
        return os.path.join(cls.OUTPUT_DIR_BASE, f"{model_name}_{dataset_name}")

os.makedirs(DetectorConfig.OUTPUT_DIR_BASE, exist_ok=True)
os.makedirs(DetectorConfig.RESULTS_DIR, exist_ok=True)

print("="*70)
print("DETECTOR CONFIGURATION")
print("="*70)
print(f"Models     : {list(DetectorConfig.MODELS.keys())}")
print(f"Epochs     : {DetectorConfig.NUM_EPOCHS}")
print(f"LR         : {DetectorConfig.LEARNING_RATE}")
print(f"Warmup     : {DetectorConfig.WARMUP_RATIO * 100}% of steps")
print(f"Dropout    : {DetectorConfig.DROPOUT}")
print(f"Batch      : {DetectorConfig.BATCH_SIZE_TRAIN}")
print(f"FP16       : {DetectorConfig.FP16}")
print(f"Force      : {DetectorConfig.FORCE_RETRAIN}")
print("="*70)


# ===============================
# CELL 2.5: CHECKPOINT UTILITIES
# ===============================

def check_model_exists(output_dir):
    """
    For DeBERTa-style training, model is saved directly to root dir.
    Just check if model weights exist in root dir.
    """
    if not os.path.exists(output_dir):
        return False, None

    model_files = ['pytorch_model.bin', 'model.safetensors']
    for mf in model_files:
        if os.path.exists(os.path.join(output_dir, mf)):
            print(f"   ✅ Found existing model in root: {output_dir}")
            return True, output_dir

    return False, None


def load_trained_model(model_checkpoint_path, base_model_name):
    print(f"   ⏳ Loading trained model from: {model_checkpoint_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_path)
    model     = AutoModelForSequenceClassification.from_pretrained(
                    model_checkpoint_path)
    print(f"   ✅ Loaded successfully")
    return model, tokenizer


def save_checkpoint_status(output_dir, status_info):
    status_file = os.path.join(output_dir, "training_status.json")
    with open(status_file, 'w') as f:
        json.dump(status_info, f, indent=2)


print("✅ Checkpoint utilities defined")


# ===============================
# CELL 3: LOAD & PREPARE DATA
# ===============================

print("\n" + "="*70)
print("LOADING DATA")
print("="*70)

hc3_train  = pd.read_csv(DetectorConfig.HC3_TRAIN)
hc3_test   = pd.read_csv(DetectorConfig.HC3_TEST)
eli5_train = pd.read_csv(DetectorConfig.ELI5_TRAIN)
eli5_test  = pd.read_csv(DetectorConfig.ELI5_TEST)

print(f"✅ Data loaded:")
print(f"   HC3:  Train={len(hc3_train):,} | Test={len(hc3_test):,}")
print(f"   ELI5: Train={len(eli5_train):,} | Test={len(eli5_test):,}")

def encode_labels(labels):
    return (labels == 'llm').astype(int)

hc3_train['label_encoded']  = encode_labels(hc3_train['label'])
hc3_test['label_encoded']   = encode_labels(hc3_test['label'])
eli5_train['label_encoded'] = encode_labels(eli5_train['label'])
eli5_test['label_encoded']  = encode_labels(eli5_test['label'])

hc3_train_split, hc3_val_split = train_test_split(
    hc3_train, test_size=DetectorConfig.VAL_SPLIT_RATIO,
    random_state=42, stratify=hc3_train['label_encoded'])

eli5_train_split, eli5_val_split = train_test_split(
    eli5_train, test_size=DetectorConfig.VAL_SPLIT_RATIO,
    random_state=42, stratify=eli5_train['label_encoded'])

print(f"✅ Splits created:")
print(f"   HC3:  Train={len(hc3_train_split):,} | Val={len(hc3_val_split):,}")
print(f"   ELI5: Train={len(eli5_train_split):,} | Val={len(eli5_val_split):,}")


# ===============================
# CELL 4: DATASET CLASS
# ===============================

class TextDetectionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text  = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text, max_length=self.max_length,
            padding='max_length', truncation=True,
            return_tensors='pt')
        return {
            'input_ids':      encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels':         torch.tensor(label, dtype=torch.long)
        }

print("✅ Dataset class defined")


# ===============================
# CELL 5: METRICS
# ===============================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs       = torch.nn.functional.softmax(
                    torch.tensor(logits), dim=-1).numpy()
    predictions = probs[:, 1]
    pred_labels = (predictions > 0.5).astype(int)
    return {
        'roc_auc':     roc_auc_score(labels, predictions),
        'brier_score': brier_score_loss(labels, predictions),
        'accuracy':    accuracy_score(labels, pred_labels)
    }

print("✅ Metrics defined")


# ===============================
# CELL 7: TRAINING FUNCTION
# no checkpointing, save final weights to root dir
# ===============================

def train_and_evaluate_detector(
    model_name,
    model_checkpoint,
    train_data,
    val_data,
    test_data_dict,
    output_dir,
    force_retrain=False
):
    print(f"\n{'#'*70}")
    print(f"TRAINING: {model_name}")
    print(f"{'#'*70}")

    # ── Checkpoint check ─────────────────────────────────────
    model_exists, checkpoint_path = check_model_exists(output_dir)

    if model_exists and not force_retrain and not DetectorConfig.FORCE_RETRAIN:
        print(f"\n🔄 Found existing model — loading and skipping training")
        try:
            model, tokenizer = load_trained_model(checkpoint_path, model_checkpoint)
            model.to(device)
            skip_training = True
        except Exception as e:
            print(f"   ⚠️ Load failed: {e} — retraining from scratch")
            skip_training = False
    else:
        print(f"\n🆕 Training from scratch")
        skip_training = False

    # ── Training ─────────────────────────────────────────────
    if not skip_training:
        print(f"\n⏳ Loading {model_checkpoint} ...")
        tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        model     = AutoModelForSequenceClassification.from_pretrained(
            model_checkpoint,
            num_labels=2,
            problem_type="single_label_classification",
            hidden_dropout_prob=DetectorConfig.DROPOUT,
            attention_probs_dropout_prob=DetectorConfig.DROPOUT
        )
        print(f"✅ Model loaded")

        train_dataset = TextDetectionDataset(
            train_data['text'].tolist(),
            train_data['label_encoded'].tolist(),
            tokenizer, DetectorConfig.MAX_SEQ_LENGTH)

        val_dataset = TextDetectionDataset(
            val_data['text'].tolist(),
            val_data['label_encoded'].tolist(),
            tokenizer, DetectorConfig.MAX_SEQ_LENGTH)

        # ── NO checkpointing, NO early stopping ──
        # Runs full epoch, saves final in-memory weights to root dir
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=DetectorConfig.NUM_EPOCHS,
            per_device_train_batch_size=DetectorConfig.BATCH_SIZE_TRAIN,
            per_device_eval_batch_size=DetectorConfig.BATCH_SIZE_EVAL,
            learning_rate=DetectorConfig.LEARNING_RATE,
            warmup_ratio=DetectorConfig.WARMUP_RATIO,
            weight_decay=DetectorConfig.WEIGHT_DECAY,
            logging_steps=DetectorConfig.LOGGING_STEPS,

            # ── Eval during training for visibility, but no checkpointing ──
            eval_strategy="steps",
            eval_steps=200,                  # print metrics every 200 steps
            save_strategy="no",              # still no checkpoint subdirs saved
            load_best_model_at_end=False,    # still use final in-memory weights

            fp16=DetectorConfig.FP16,
            dataloader_num_workers=2,
            report_to="none",
            disable_tqdm=False
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            # No EarlyStoppingCallback
        )

        print(f"\n Starting training ...")
        print(f"   Train samples : {len(train_dataset):,}")
        print(f"   Val samples   : {len(val_dataset):,}")
        print(f"   Epochs        : {DetectorConfig.NUM_EPOCHS}")
        print(f"   Batch size    : {DetectorConfig.BATCH_SIZE_TRAIN}")
        print(f"   LR            : {DetectorConfig.LEARNING_RATE}")
        print(f"   Warmup        : {DetectorConfig.WARMUP_RATIO*100}% of steps")

        try:
            trainer.train()
            print(f"\n✅ Training complete!")

            # ── Save final weights directly to root dir ──────
            print(f"⏳ Saving model to: {output_dir}")
            trainer.save_model(output_dir)
            tokenizer.save_pretrained(output_dir)
            print(f"✅ Saved to root dir — no checkpoint subdirs")

            # ── Post-training sanity check ───────────────────
            print("\n⏳ Post-training sanity check ...")
            dummy_texts = [
                "yeah i just grabbed some lunch and honestly it was pretty mid tbh",
                "Certainly! The mitochondria is the powerhouse of the cell, responsible for producing adenosine triphosphate (ATP) through the process of cellular respiration, which involves glycolysis, the Krebs cycle, and oxidative phosphorylation."
            ]
            dummy_enc = tokenizer(
                dummy_texts, return_tensors="pt",
                padding=True, truncation=True).to(device)
            with torch.no_grad():
                dummy_logits = model(**dummy_enc).logits
            dummy_probs = torch.softmax(dummy_logits, dim=-1)[:, 1].cpu().numpy()
            dummy_std   = dummy_probs.std()

            print(f"   Sanity scores : {dummy_probs.round(4)}")
            print(f"   Score std     : {dummy_std:.4f}")

            if dummy_std < 0.05:
                print(f"   ⚠️  WARNING: Model collapsed (std={dummy_std:.4f})")
                print(f"   Check training logs — model may not have converged")
            else:
                print(f"   ✅ Model looks healthy")

            save_checkpoint_status(output_dir, {
                'model_name':    model_name,
                'status':        'completed',
                'output_dir':    output_dir,
                'sanity_std':    float(dummy_std),
                'sanity_scores': dummy_probs.tolist(),
                'collapsed':     bool(dummy_std < 0.05),
                'timestamp':     pd.Timestamp.now().isoformat()
            })

        except Exception as e:
            print(f"\n❌ Training failed: {e}")
            save_checkpoint_status(output_dir, {
                'model_name': model_name,
                'status':     'failed',
                'error':      str(e),
                'timestamp':  pd.Timestamp.now().isoformat()
            })
            raise

    else:
        # Dummy trainer for prediction only
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_eval_batch_size=DetectorConfig.BATCH_SIZE_EVAL,
            fp16=DetectorConfig.FP16,
            report_to="none"
        )
        trainer = Trainer(
            model=model,
            args=training_args,
            compute_metrics=compute_metrics
        )

    # ── Evaluate on test sets ─────────────────────────────────
    results = {}
    for test_name, test_data in test_data_dict.items():
        print(f"\n📊 Evaluating on {test_name} ...")
        test_dataset = TextDetectionDataset(
            test_data['text'].tolist(),
            test_data['label_encoded'].tolist(),
            tokenizer, DetectorConfig.MAX_SEQ_LENGTH)

        predictions        = trainer.predict(test_dataset)
        logits             = predictions.predictions
        probs              = torch.nn.functional.softmax(
                                torch.tensor(logits), dim=-1).numpy()
        detectability_scores = probs[:, 1]

        y_true = test_data['label_encoded'].values
        y_pred = (detectability_scores > 0.5).astype(int)

        results[test_name] = {
            'y_true':               y_true,
            'y_pred':               y_pred,
            'detectability_scores': detectability_scores,
            'roc_auc':    roc_auc_score(y_true, detectability_scores),
            'brier_score':brier_score_loss(y_true, detectability_scores),
            'log_loss':   log_loss(y_true, detectability_scores),
            'accuracy':   accuracy_score(y_true, y_pred)
        }
        print(f"   ROC-AUC  : {results[test_name]['roc_auc']:.4f}")
        print(f"   Brier    : {results[test_name]['brier_score']:.4f}")
        print(f"   Accuracy : {results[test_name]['accuracy']:.4f}")

    del model, trainer
    torch.cuda.empty_cache()
    return results

print("✅ Training function defined")


# ===============================
# CELL 8: BERT - HC3
# ===============================
print("\n" + "#"*70)
print("BERT DETECTOR - HC3")
print("#"*70)
try:
    bert_hc3_results = train_and_evaluate_detector(
        model_name="BERT_HC3",
        model_checkpoint=DetectorConfig.MODELS['BERT'],
        train_data=hc3_train_split,
        val_data=hc3_val_split,
        test_data_dict={'hc3_to_hc3': hc3_test, 'hc3_to_eli5': eli5_test},
        output_dir=DetectorConfig.get_output_dir('BERT', 'hc3'))
except Exception as e:
    print(f"❌ BERT_HC3 failed: {e}")
    bert_hc3_results = None


# ===============================
# CELL 9: BERT - ELI5
# ===============================
print("\n" + "#"*70)
print("BERT DETECTOR - ELI5")
print("#"*70)
try:
    bert_eli5_results = train_and_evaluate_detector(
        model_name="BERT_ELI5",
        model_checkpoint=DetectorConfig.MODELS['BERT'],
        train_data=eli5_train_split,
        val_data=eli5_val_split,
        test_data_dict={'eli5_to_eli5': eli5_test, 'eli5_to_hc3': hc3_test},
        output_dir=DetectorConfig.get_output_dir('BERT', 'eli5'))
except Exception as e:
    print(f"❌ BERT_ELI5 failed: {e}")
    bert_eli5_results = None


# ===============================
# CELL 10: RoBERTa - HC3
# ===============================
print("\n" + "#"*70)
print("RoBERTa DETECTOR - HC3")
print("#"*70)
try:
    roberta_hc3_results = train_and_evaluate_detector(
        model_name="RoBERTa_HC3",
        model_checkpoint=DetectorConfig.MODELS['RoBERTa'],
        train_data=hc3_train_split,
        val_data=hc3_val_split,
        test_data_dict={'hc3_to_hc3': hc3_test, 'hc3_to_eli5': eli5_test},
        output_dir=DetectorConfig.get_output_dir('RoBERTa', 'hc3'))
except Exception as e:
    print(f"❌ RoBERTa_HC3 failed: {e}")
    roberta_hc3_results = None


# ===============================
# CELL 11: RoBERTa - ELI5
# ===============================
print("\n" + "#"*70)
print("RoBERTa DETECTOR - ELI5")
print("#"*70)
try:
    roberta_eli5_results = train_and_evaluate_detector(
        model_name="RoBERTa_ELI5",
        model_checkpoint=DetectorConfig.MODELS['RoBERTa'],
        train_data=eli5_train_split,
        val_data=eli5_val_split,
        test_data_dict={'eli5_to_eli5': eli5_test, 'eli5_to_hc3': hc3_test},
        output_dir=DetectorConfig.get_output_dir('RoBERTa', 'eli5'))
except Exception as e:
    print(f"❌ RoBERTa_ELI5 failed: {e}")
    roberta_eli5_results = None


# ===============================
# CELL 12: ELECTRA - HC3
# ===============================
print("\n" + "#"*70)
print("ELECTRA DETECTOR - HC3")
print("#"*70)
try:
    electra_hc3_results = train_and_evaluate_detector(
        model_name="ELECTRA_HC3",
        model_checkpoint=DetectorConfig.MODELS['ELECTRA'],
        train_data=hc3_train_split,
        val_data=hc3_val_split,
        test_data_dict={'hc3_to_hc3': hc3_test, 'hc3_to_eli5': eli5_test},
        output_dir=DetectorConfig.get_output_dir('ELECTRA', 'hc3'))
except Exception as e:
    print(f"❌ ELECTRA_HC3 failed: {e}")
    electra_hc3_results = None


# ===============================
# CELL 13: ELECTRA - ELI5
# ===============================
print("\n" + "#"*70)
print("ELECTRA DETECTOR - ELI5")
print("#"*70)
try:
    electra_eli5_results = train_and_evaluate_detector(
        model_name="ELECTRA_ELI5",
        model_checkpoint=DetectorConfig.MODELS['ELECTRA'],
        train_data=eli5_train_split,
        val_data=eli5_val_split,
        test_data_dict={'eli5_to_eli5': eli5_test, 'eli5_to_hc3': hc3_test},
        output_dir=DetectorConfig.get_output_dir('ELECTRA', 'eli5'))
except Exception as e:
    print(f"❌ ELECTRA_ELI5 failed: {e}")
    electra_eli5_results = None


# ===============================
# CELL 16: CONSOLIDATE RESULTS
# ===============================
print("\n" + "="*70)
print("CONSOLIDATING RESULTS")
print("="*70)

successful_models = []
failed_models     = []
neural_results    = {}

for name, hc3_r, eli5_r in [
    ('BERT',    bert_hc3_results,    bert_eli5_results),
    ('RoBERTa', roberta_hc3_results, roberta_eli5_results),
    ('ELECTRA', electra_hc3_results, electra_eli5_results),
]:
    if hc3_r is not None and eli5_r is not None:
        neural_results[name] = {**hc3_r, **eli5_r}
        successful_models.append(name)
    else:
        failed_models.append(name)

print(f"✅ Successful : {successful_models}")
if failed_models:
    print(f"❌ Failed     : {failed_models}")
print(f"✅ Consolidated {len(neural_results)} models")


# ===============================
# CELL 17: SCORE DISTRIBUTIONS
# ===============================
if neural_results:
    n = len(neural_results)
    fig, axes = plt.subplots(n, 4, figsize=(20, 5*n))
    if n == 1: axes = axes.reshape(1, -1)
    fig.suptitle('Neural Detectors: Score Distributions',
                 fontsize=18, fontweight='bold')

    for i, (mname, mresults) in enumerate(neural_results.items()):
        for j, (ename, edata) in enumerate(mresults.items()):
            ax = axes[i, j]
            hs = edata['detectability_scores'][edata['y_true'] == 0]
            ls = edata['detectability_scores'][edata['y_true'] == 1]
            ax.hist(hs, bins=30, alpha=0.6, label='Human',
                    color='#3498db', range=(0,1))
            ax.hist(ls, bins=30, alpha=0.6, label='LLM',
                    color='#e74c3c', range=(0,1))
            ax.axvline(0.5, color='black', ls='--', alpha=0.4)
            ax.set_title(f'{mname} | {ename}\nAUC={edata["roc_auc"]:.3f}',
                         fontsize=10, fontweight='bold')
            ax.set_xlim(0, 1)
            ax.legend(fontsize=8)
            ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(f"{DetectorConfig.RESULTS_DIR}/score_distributions.png",
                dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Saved: score_distributions.png")


# ===============================
# CELL 18: CALIBRATION CURVES
# ===============================
if neural_results:
    n = len(neural_results)
    fig, axes = plt.subplots(n, 4, figsize=(20, 5*n))
    if n == 1: axes = axes.reshape(1, -1)
    fig.suptitle('Neural Detectors: Calibration Curves',
                 fontsize=18, fontweight='bold')

    for i, (mname, mresults) in enumerate(neural_results.items()):
        for j, (ename, edata) in enumerate(mresults.items()):
            ax = axes[i, j]
            fop, mpv = calibration_curve(
                edata['y_true'], edata['detectability_scores'],
                n_bins=10, strategy='uniform')
            ax.plot(mpv, fop, 's-', label='Detector',
                    linewidth=2.5, color='#2ecc71')
            ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Perfect')
            ax.set_title(f'{mname} | {ename}', fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(alpha=0.3)
            ax.set_xlim(0,1); ax.set_ylim(0,1)
    plt.tight_layout()
    plt.savefig(f"{DetectorConfig.RESULTS_DIR}/calibration_curves.png",
                dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Saved: calibration_curves.png")


# ===============================
# CELL 19: ROC CURVES
# ===============================
if neural_results:
    n = len(neural_results)
    fig, axes = plt.subplots(n, 4, figsize=(20, 5*n))
    if n == 1: axes = axes.reshape(1, -1)
    fig.suptitle('Neural Detectors: ROC Curves',
                 fontsize=18, fontweight='bold')

    for i, (mname, mresults) in enumerate(neural_results.items()):
        for j, (ename, edata) in enumerate(mresults.items()):
            ax = axes[i, j]
            fpr, tpr, _ = roc_curve(
                edata['y_true'], edata['detectability_scores'])
            ax.plot(fpr, tpr, linewidth=2.5,
                    label=f"AUC={edata['roc_auc']:.3f}", color='#9b59b6')
            ax.plot([0,1],[0,1],'k--', alpha=0.4)
            ax.set_title(f'{mname} | {ename}', fontsize=10)
            ax.legend(fontsize=8, loc='lower right')
            ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{DetectorConfig.RESULTS_DIR}/roc_curves.png",
                dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Saved: roc_curves.png")


# ===============================
# CELL 21: PERFORMANCE SUMMARY
# ===============================
if neural_results:
    rows = []
    for mname, mresults in neural_results.items():
        for ename, edata in mresults.items():
            hs = edata['detectability_scores'][edata['y_true'] == 0]
            ls = edata['detectability_scores'][edata['y_true'] == 1]
            rows.append({
                'Detector':       mname,
                'Evaluation':     ename,
                'ROC-AUC':        edata['roc_auc'],
                'Accuracy':       edata['accuracy'],
                'Brier Score':    edata['brier_score'],
                'Log Loss':       edata['log_loss'],
                'Mean Human':     hs.mean(),
                'Mean LLM':       ls.mean(),
                'Separation':     ls.mean() - hs.mean()
            })
    summary_df = pd.DataFrame(rows)
    print(summary_df.round(4).to_string(index=False))
    summary_df.to_csv(f"{DetectorConfig.RESULTS_DIR}/performance_summary.csv",
                      index=False)
    print("\n✅ Saved: performance_summary.csv")


# ===============================
# CELL 22: SAVE RESULTS
# ===============================
if neural_results:
    with open(f"{DetectorConfig.RESULTS_DIR}/neural_detector_results.pkl", "wb") as f:
        pickle.dump(neural_results, f)
    print("✅ Saved: neural_detector_results.pkl")

print("\n" + "="*70)
print("✅ TRAINING COMPLETE")
print("="*70)
print(f"✅ Successful : {successful_models}")
if failed_models:
    print(f"❌ Failed     : {failed_models}")
print(f"📊 Results in : {DetectorConfig.RESULTS_DIR}")

In [ ]:
# ===============================
# DIAGNOSTIC CELL: REAL MODEL CHECK
# ===============================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODELS_TO_CHECK = {
    "BERT_hc3":    "./models/BERT_hc3",
    "RoBERTa_hc3": "./models/RoBERTa_hc3",
    "ELECTRA_hc3": "./models/ELECTRA_hc3",
}

# Use real samples from actual test set
hc3_test = pd.read_csv("hc3_test.csv")
hc3_test['label_encoded'] = (hc3_test['label'] == 'llm').astype(int)

# Take 100 human + 100 llm samples
human_samples = hc3_test[hc3_test['label'] == 'human'].sample(100, random_state=42)
llm_samples   = hc3_test[hc3_test['label'] == 'llm'].sample(100, random_state=42)
probe_df      = pd.concat([human_samples, llm_samples]).reset_index(drop=True)

print("="*60)
print("REAL DIAGNOSTIC CHECK — 100 human + 100 LLM from hc3_test")
print("="*60)

for model_key, model_path in MODELS_TO_CHECK.items():
    print(f"\n{'─'*60}")
    print(f"Checking: {model_key}")
    print(f"Path    : {model_path}")

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model     = AutoModelForSequenceClassification.from_pretrained(model_path)
        model.to(device)
        model.eval()

        all_probs = []
        batch_size = 32

        for i in range(0, len(probe_df), batch_size):
            batch_texts = probe_df['text'].iloc[i:i+batch_size].tolist()
            enc = tokenizer(
                batch_texts, return_tensors="pt",
                padding=True, truncation=True,
                max_length=512).to(device)
            with torch.no_grad():
                logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)

        import numpy as np
        all_probs = np.array(all_probs)
        y_true    = probe_df['label_encoded'].values

        auc        = roc_auc_score(y_true, all_probs)
        mean_human = all_probs[y_true == 0].mean()
        mean_llm   = all_probs[y_true == 1].mean()
        std_all    = all_probs.std()

        print(f"  ROC-AUC        : {auc:.4f}")
        print(f"  Mean P(LLM) — Human texts : {mean_human:.4f}")
        print(f"  Mean P(LLM) — LLM texts   : {mean_llm:.4f}")
        print(f"  Score std (all): {std_all:.4f}")

        if auc > 0.7:
            print(f"  ✅ MODEL IS WORKING — AUC={auc:.4f}")
        elif auc < 0.4:
            print(f"  ❌ INVERTED — predicting backwards (AUC={auc:.4f})")
        else:
            print(f"  ⚠️  WEAK but not collapsed (AUC={auc:.4f})")

        del model, tokenizer
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"  ❌ LOAD FAILED: {e}")

In [ ]:
# ===============================
# DISTILBERT DETECTOR TRAINING
# no checkpointing, full epoch, save to root dir
# ===============================

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, log_loss,
    roc_curve, accuracy_score
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import json
import os
import pickle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("✅ All imports loaded")


# ===============================
# CONFIGURATION
# ===============================

class DistilBERTConfig:
    MODEL_NAME = 'distilbert-base-uncased'

    HC3_TRAIN  = "hc3_train.csv"
    HC3_TEST   = "hc3_test.csv"
    ELI5_TRAIN = "eli5_train.csv"
    ELI5_TEST  = "eli5_test.csv"

    NUM_EPOCHS         = 1
    BATCH_SIZE_TRAIN   = 32
    BATCH_SIZE_EVAL    = 64
    LEARNING_RATE      = 2e-5
    WARMUP_RATIO       = 0.06
    WEIGHT_DECAY       = 0.01
    DROPOUT            = 0.2
    ATTENTION_DROPOUT  = 0.2
    VAL_SPLIT_RATIO    = 0.1
    LOGGING_STEPS      = 100
    MAX_SEQ_LENGTH     = 512
    FP16               = torch.cuda.is_available()

    OUTPUT_DIR_BASE = "./models"
    RESULTS_DIR     = "./results"

    @classmethod
    def get_output_dir(cls, dataset_name):
        return os.path.join(cls.OUTPUT_DIR_BASE, f"DistilBERT_{dataset_name}")

os.makedirs(DistilBERTConfig.OUTPUT_DIR_BASE, exist_ok=True)
os.makedirs(DistilBERTConfig.RESULTS_DIR, exist_ok=True)

print("="*70)
print("DISTILBERT CONFIGURATION")
print("="*70)
print(f"Model        : {DistilBERTConfig.MODEL_NAME}")
print(f"Epochs       : {DistilBERTConfig.NUM_EPOCHS}")
print(f"LR           : {DistilBERTConfig.LEARNING_RATE}")
print(f"Warmup       : {DistilBERTConfig.WARMUP_RATIO*100}% of steps")
print(f"Dropout      : {DistilBERTConfig.DROPOUT}")
print(f"Batch size   : {DistilBERTConfig.BATCH_SIZE_TRAIN}")
print(f"FP16         : {DistilBERTConfig.FP16}")
print("="*70)


# ===============================
# LOAD DATA
# ===============================

hc3_train  = pd.read_csv(DistilBERTConfig.HC3_TRAIN)
hc3_test   = pd.read_csv(DistilBERTConfig.HC3_TEST)
eli5_train = pd.read_csv(DistilBERTConfig.ELI5_TRAIN)
eli5_test  = pd.read_csv(DistilBERTConfig.ELI5_TEST)

def encode_labels(labels):
    return (labels == 'llm').astype(int)

hc3_train['label_encoded']  = encode_labels(hc3_train['label'])
hc3_test['label_encoded']   = encode_labels(hc3_test['label'])
eli5_train['label_encoded'] = encode_labels(eli5_train['label'])
eli5_test['label_encoded']  = encode_labels(eli5_test['label'])

hc3_train_split, hc3_val_split = train_test_split(
    hc3_train, test_size=DistilBERTConfig.VAL_SPLIT_RATIO,
    random_state=42, stratify=hc3_train['label_encoded'])

eli5_train_split, eli5_val_split = train_test_split(
    eli5_train, test_size=DistilBERTConfig.VAL_SPLIT_RATIO,
    random_state=42, stratify=eli5_train['label_encoded'])

print(f"✅ Data loaded:")
print(f"   HC3:  Train={len(hc3_train_split):,} | Val={len(hc3_val_split):,} | Test={len(hc3_test):,}")
print(f"   ELI5: Train={len(eli5_train_split):,} | Val={len(eli5_val_split):,} | Test={len(eli5_test):,}")


# ===============================
# DATASET CLASS
# ===============================

class TextDetectionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length', truncation=True,
            return_tensors='pt')
        return {
            'input_ids':      encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

print("✅ Dataset class defined")


# ===============================
# METRICS
# ===============================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs       = torch.nn.functional.softmax(
                    torch.tensor(logits), dim=-1).numpy()
    predictions = probs[:, 1]
    pred_labels = (predictions > 0.5).astype(int)
    return {
        'roc_auc':     roc_auc_score(labels, predictions),
        'brier_score': brier_score_loss(labels, predictions),
        'accuracy':    accuracy_score(labels, pred_labels)
    }

print("✅ Metrics defined")


# ===============================
# TRAINING FUNCTION
# ===============================

def train_distilbert_detector(train_data, val_data, test_data_dict,
                               output_dir, dataset_name):
    print(f"\n{'#'*70}")
    print(f"TRAINING: DistilBERT on {dataset_name}")
    print(f"{'#'*70}")

    os.makedirs(output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(DistilBERTConfig.MODEL_NAME)

    # DistilBERT-specific dropout param names
    model = AutoModelForSequenceClassification.from_pretrained(
        DistilBERTConfig.MODEL_NAME,
        num_labels=2,
        problem_type="single_label_classification",
        dropout=DistilBERTConfig.DROPOUT,
        attention_dropout=DistilBERTConfig.ATTENTION_DROPOUT
    )
    print(f"✅ Model loaded — dropout={DistilBERTConfig.DROPOUT}")

    train_dataset = TextDetectionDataset(
        train_data['text'].tolist(),
        train_data['label_encoded'].tolist(),
        tokenizer, DistilBERTConfig.MAX_SEQ_LENGTH)

    val_dataset = TextDetectionDataset(
        val_data['text'].tolist(),
        val_data['label_encoded'].tolist(),
        tokenizer, DistilBERTConfig.MAX_SEQ_LENGTH)

    # ── no checkpointing, no early stopping ──
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=DistilBERTConfig.NUM_EPOCHS,
        per_device_train_batch_size=DistilBERTConfig.BATCH_SIZE_TRAIN,
        per_device_eval_batch_size=DistilBERTConfig.BATCH_SIZE_EVAL,
        learning_rate=DistilBERTConfig.LEARNING_RATE,
        warmup_ratio=DistilBERTConfig.WARMUP_RATIO,
        weight_decay=DistilBERTConfig.WEIGHT_DECAY,
        logging_steps=DistilBERTConfig.LOGGING_STEPS,

        # eval during training for visibility only
        eval_strategy="steps",
        eval_steps=200,

        # no checkpointing
        save_strategy="no",
        load_best_model_at_end=False,

        fp16=DistilBERTConfig.FP16,
        dataloader_num_workers=2,
        report_to="none",
        disable_tqdm=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,

    )

    print(f"\n Starting training ...")
    print(f"   Train samples : {len(train_dataset):,}")
    print(f"   Val samples   : {len(val_dataset):,}")
    print(f"   Epochs        : {DistilBERTConfig.NUM_EPOCHS}")
    print(f"   Batch size    : {DistilBERTConfig.BATCH_SIZE_TRAIN}")
    print(f"   LR            : {DistilBERTConfig.LEARNING_RATE}")

    trainer.train()
    print(f"\n✅ Training complete!")

    # ── Save final in-memory weights directly to root dir ──
    print(f"⏳ Saving model to: {output_dir}")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"✅ Saved to root dir — no checkpoint subdirs")

    # ── Sanity check using real test samples ────────────────
    print("\n⏳ Post-training sanity check ...")
    hc3_test_loaded = pd.read_csv(DistilBERTConfig.HC3_TEST)
    hc3_test_loaded['label_encoded'] = encode_labels(hc3_test_loaded['label'])
    human_sample = hc3_test_loaded[hc3_test_loaded['label'] == 'human']['text'].iloc[0][:300]
    llm_sample   = hc3_test_loaded[hc3_test_loaded['label'] == 'llm']['text'].iloc[0][:300]

    dummy_enc = tokenizer(
        [human_sample, llm_sample],
        return_tensors="pt", padding=True,
        truncation=True, max_length=512).to(device)

    model.to(device)
    model.eval()
    with torch.no_grad():
        dummy_logits = model(**dummy_enc).logits
    dummy_probs = torch.softmax(dummy_logits, dim=-1)[:, 1].cpu().numpy()
    dummy_std   = dummy_probs.std()

    print(f"   Human score   : {dummy_probs[0]:.4f}")
    print(f"   LLM score     : {dummy_probs[1]:.4f}")
    print(f"   Score std     : {dummy_std:.4f}")

    if dummy_std < 0.05:
        print(f"   ⚠️  WARNING: Model may be collapsed (std={dummy_std:.4f})")
    else:
        print(f"   ✅ Model looks healthy")

    with open(os.path.join(output_dir, "training_status.json"), 'w') as f:
        json.dump({
            'model_name':    f'DistilBERT_{dataset_name}',
            'status':        'completed',
            'sanity_std':    float(dummy_std),
            'sanity_scores': dummy_probs.tolist(),
            'collapsed':     bool(dummy_std < 0.05),
            'timestamp':     pd.Timestamp.now().isoformat()
        }, f, indent=2)

    # ── Evaluate on test sets ────────────────────────────────
    results = {}
    for test_name, test_data in test_data_dict.items():
        print(f"\n📊 Evaluating on {test_name} ...")
        test_dataset = TextDetectionDataset(
            test_data['text'].tolist(),
            test_data['label_encoded'].tolist(),
            tokenizer, DistilBERTConfig.MAX_SEQ_LENGTH)

        predictions          = trainer.predict(test_dataset)
        probs                = torch.nn.functional.softmax(
                                torch.tensor(predictions.predictions),
                                dim=-1).numpy()
        detectability_scores = probs[:, 1]
        y_true               = test_data['label_encoded'].values
        y_pred               = (detectability_scores > 0.5).astype(int)

        results[test_name] = {
            'y_true':               y_true,
            'y_pred':               y_pred,
            'detectability_scores': detectability_scores,
            'roc_auc':    roc_auc_score(y_true, detectability_scores),
            'brier_score':brier_score_loss(y_true, detectability_scores),
            'log_loss':   log_loss(y_true, detectability_scores),
            'accuracy':   accuracy_score(y_true, y_pred)
        }
        print(f"   ROC-AUC  : {results[test_name]['roc_auc']:.4f}")
        print(f"   Brier    : {results[test_name]['brier_score']:.4f}")
        print(f"   Accuracy : {results[test_name]['accuracy']:.4f}")

    del model, trainer
    torch.cuda.empty_cache()
    return results


# ===============================
# TRAIN HC3
# ===============================

distilbert_hc3_results = train_distilbert_detector(
    train_data=hc3_train_split,
    val_data=hc3_val_split,
    test_data_dict={'hc3_to_hc3': hc3_test, 'hc3_to_eli5': eli5_test},
    output_dir=DistilBERTConfig.get_output_dir('hc3'),
    dataset_name='HC3')


# ===============================
# TRAIN ELI5
# ===============================

distilbert_eli5_results = train_distilbert_detector(
    train_data=eli5_train_split,
    val_data=eli5_val_split,
    test_data_dict={'eli5_to_eli5': eli5_test, 'eli5_to_hc3': hc3_test},
    output_dir=DistilBERTConfig.get_output_dir('eli5'),
    dataset_name='ELI5')


# ===============================
# CONSOLIDATE & VISUALIZE
# ===============================

distilbert_results = {
    'DistilBERT': {**distilbert_hc3_results, **distilbert_eli5_results}
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('DistilBERT: Score Distributions', fontsize=18, fontweight='bold')
for j, (ename, edata) in enumerate(distilbert_results['DistilBERT'].items()):
    ax = axes[j]
    hs = edata['detectability_scores'][edata['y_true'] == 0]
    ls = edata['detectability_scores'][edata['y_true'] == 1]
    ax.hist(hs, bins=30, alpha=0.6, label='Human', color='#3498db', range=(0,1))
    ax.hist(ls, bins=30, alpha=0.6, label='LLM',   color='#e74c3c', range=(0,1))
    ax.axvline(0.5, color='black', ls='--', alpha=0.4)
    ax.set_title(f'DistilBERT | {ename}\nAUC={edata["roc_auc"]:.3f}', fontsize=10)
    ax.set_xlim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f"{DistilBERTConfig.RESULTS_DIR}/distilbert_score_distributions.png",
            dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('DistilBERT: ROC Curves', fontsize=18, fontweight='bold')
for j, (ename, edata) in enumerate(distilbert_results['DistilBERT'].items()):
    ax = axes[j]
    fpr, tpr, _ = roc_curve(edata['y_true'], edata['detectability_scores'])
    ax.plot(fpr, tpr, linewidth=2.5,
            label=f"AUC={edata['roc_auc']:.3f}", color='#9b59b6')
    ax.plot([0,1],[0,1],'k--', alpha=0.4)
    ax.set_title(f'DistilBERT | {ename}', fontsize=10)
    ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{DistilBERTConfig.RESULTS_DIR}/distilbert_roc_curves.png",
            dpi=150, bbox_inches='tight')
plt.show()

# Summary table
rows = []
for ename, edata in distilbert_results['DistilBERT'].items():
    hs = edata['detectability_scores'][edata['y_true'] == 0]
    ls = edata['detectability_scores'][edata['y_true'] == 1]
    rows.append({
        'Evaluation':  ename,
        'ROC-AUC':     edata['roc_auc'],
        'Accuracy':    edata['accuracy'],
        'Brier Score': edata['brier_score'],
        'Log Loss':    edata['log_loss'],
        'Mean Human':  hs.mean(),
        'Mean LLM':    ls.mean(),
        'Separation':  ls.mean() - hs.mean()
    })
summary_df = pd.DataFrame(rows)
print("\n" + "="*70)
print("DISTILBERT PERFORMANCE SUMMARY")
print("="*70)
print(summary_df.round(4).to_string(index=False))
summary_df.to_csv(f"{DistilBERTConfig.RESULTS_DIR}/distilbert_performance_summary.csv",
                  index=False)

with open(f"{DistilBERTConfig.RESULTS_DIR}/distilbert_results.pkl", "wb") as f:
    pickle.dump(distilbert_results, f)

print("\n✅ DISTILBERT TRAINING COMPLETE")
print(f"📊 Results saved to: {DistilBERTConfig.RESULTS_DIR}")

In [ ]:
# ================================================================
# DeBERTa-v3-base — Transformer Family Extension
# ================================================================
# ── Cell 1: Install & Imports ──────────────────────────────────
!pip install -q transformers datasets accelerate sentencepiece

import pandas as pd
import numpy as np
import torch
import shutil, pickle, os

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, log_loss,
    accuracy_score, roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Cell 2: Config ─────────────────────────────────────────────
class DeBERTaConfig:
    MODEL_CHECKPOINT = "microsoft/deberta-v3-base"

    HC3_TRAIN  = "hc3_train.csv"
    HC3_TEST   = "hc3_test.csv"
    ELI5_TRAIN = "eli5_train.csv"
    ELI5_TEST  = "eli5_test.csv"

    NUM_EPOCHS              = 1
    BATCH_SIZE_TRAIN        = 16
    BATCH_SIZE_EVAL         = 32
    LEARNING_RATE           = 2e-5
    WARMUP_STEPS            = 500
    WEIGHT_DECAY            = 0.01
    DROPOUT                 = 0.2
    VAL_SPLIT_RATIO         = 0.1
    LOGGING_STEPS           = 100
    MAX_SEQ_LENGTH          = 512

    # ── Precision: full fp32 ──────────────────────────────────
    # bf16 silently zeroes DeBERTa-v3 gradients (disentangled
    # attention produces small gradient magnitudes that underflow
    # in bf16's 7-bit mantissa). fp16 causes unscaling crash.
    # fp32 is the only safe option for this architecture.
    FP16 = False
    BF16 = False

    # ── No checkpoint reloading ───────────────────────────────
    # load_best_model_at_end reloads a saved checkpoint whose
    # LayerNorm keys use old gamma/beta naming → all 24 LayerNorm
    # layers reinitialise to random → AUC collapses to 0.50.
    # With save_strategy="no" we never write intermediate ckpts
    # and use the final in-memory model directly. Given 1 epoch
    # and sufficient RAM this is cleaner and faster.
    SAVE_STRATEGY           = "no"
    LOAD_BEST_MODEL_AT_END  = False

    # Gradient clipping — important for DeBERTa-v3 stability
    MAX_GRAD_NORM           = 1.0

    OUTPUT_DIR_BASE = "./models"
    RESULTS_DIR     = "./results"

    @classmethod
    def get_output_dir(cls, dataset_name):
        return os.path.join(cls.OUTPUT_DIR_BASE,
                            f"DeBERTa_{dataset_name}")

os.makedirs(DeBERTaConfig.OUTPUT_DIR_BASE, exist_ok=True)
os.makedirs(DeBERTaConfig.RESULTS_DIR,     exist_ok=True)

print("\nDeBERTa-v3-base Config:")
print(f"  Checkpoint      : {DeBERTaConfig.MODEL_CHECKPOINT}")
print(f"  Epochs          : {DeBERTaConfig.NUM_EPOCHS}")
print(f"  Batch (train)   : {DeBERTaConfig.BATCH_SIZE_TRAIN}")
print(f"  LR              : {DeBERTaConfig.LEARNING_RATE}")
print(f"  Dropout         : {DeBERTaConfig.DROPOUT}")
print(f"  Precision       : fp32 (fp16=False, bf16=False)")
print(f"  Checkpointing   : DISABLED (save_strategy=no)")
print(f"  Max grad norm   : {DeBERTaConfig.MAX_GRAD_NORM}")

# ── Cell 3: Load & Split Data ──────────────────────────────────
def encode_labels(series):
    return (series == "llm").astype(int)

hc3_train_raw  = pd.read_csv(DeBERTaConfig.HC3_TRAIN)
hc3_test       = pd.read_csv(DeBERTaConfig.HC3_TEST)
eli5_train_raw = pd.read_csv(DeBERTaConfig.ELI5_TRAIN)
eli5_test      = pd.read_csv(DeBERTaConfig.ELI5_TEST)

for df in [hc3_train_raw, hc3_test, eli5_train_raw, eli5_test]:
    df["label_encoded"] = encode_labels(df["label"])

hc3_train_split, hc3_val_split = train_test_split(
    hc3_train_raw,
    test_size=DeBERTaConfig.VAL_SPLIT_RATIO,
    random_state=42,
    stratify=hc3_train_raw["label_encoded"])

eli5_train_split, eli5_val_split = train_test_split(
    eli5_train_raw,
    test_size=DeBERTaConfig.VAL_SPLIT_RATIO,
    random_state=42,
    stratify=eli5_train_raw["label_encoded"])

print(f"\nHC3  Train={len(hc3_train_split):,} | "
      f"Val={len(hc3_val_split):,} | "
      f"Test={len(hc3_test):,}")
print(f"ELI5 Train={len(eli5_train_split):,} | "
      f"Val={len(eli5_val_split):,} | "
      f"Test={len(eli5_test):,}")

# ── Cell 4: Dataset Class ──────────────────────────────────────
class TextDetectionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            # token_type_ids intentionally omitted —
            # DeBERTa-v3 does not use segment IDs
            "labels": torch.tensor(
                self.labels[idx], dtype=torch.long),
        }

# ── Cell 5: Metrics for Trainer callback ──────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.nn.functional.softmax(
                 torch.tensor(logits), dim=-1).numpy()
    scores = probs[:, 1]
    preds  = (scores > 0.5).astype(int)
    return {
        "roc_auc":     roc_auc_score(labels, scores),
        "brier_score": brier_score_loss(labels, scores),
        "accuracy":    accuracy_score(labels, preds),
    }

# ── Cell 6: Training Function ──────────────────────────────────
def train_deberta(train_data, val_data, test_data_dict,
                  output_dir, tag):
    """
    Train DeBERTa-v3-base for binary AI-text detection.

    Precision strategy:
        fp32 throughout — bf16 silently zeroes gradients for this
        architecture; fp16 crashes the grad scaler. No mixed
        precision is the safest and only reliable option here.

    Checkpointing strategy:
        Disabled entirely (save_strategy="no",
        load_best_model_at_end=False). The Trainer runs one full
        epoch and the final in-memory model is used directly for
        prediction. This eliminates the LayerNorm gamma/beta key
        mismatch that caused AUC ≈ 0.50 in previous runs.
    """
    print(f"\n{'='*60}")
    print(f"Training DeBERTa-v3-base  [dataset: {tag}]")
    print(f"{'='*60}")

    # Delete any stale checkpoints from previous failed runs
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
        print(f"🗑️  Cleared stale output dir: {output_dir}")
    os.makedirs(output_dir, exist_ok=True)

    # ── Tokenizer ─────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(
        DeBERTaConfig.MODEL_CHECKPOINT)

    # ── Model — forced to fp32 ────────────────────────────────
    model = AutoModelForSequenceClassification.from_pretrained(
        DeBERTaConfig.MODEL_CHECKPOINT,
        num_labels=2,
        hidden_dropout_prob=DeBERTaConfig.DROPOUT,
        attention_probs_dropout_prob=DeBERTaConfig.DROPOUT,
        ignore_mismatched_sizes=True,
    ).float()
    # The LOAD REPORT warnings printed above (UNEXPECTED lm_head
    # keys, MISSING pooler/classifier keys) are expected and safe.

    # ── Datasets ──────────────────────────────────────────────
    train_ds = TextDetectionDataset(
        train_data["text"].tolist(),
        train_data["label_encoded"].tolist(),
        tokenizer, DeBERTaConfig.MAX_SEQ_LENGTH)

    val_ds = TextDetectionDataset(
        val_data["text"].tolist(),
        val_data["label_encoded"].tolist(),
        tokenizer, DeBERTaConfig.MAX_SEQ_LENGTH)

    # ── TrainingArguments ─────────────────────────────────────
    args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = DeBERTaConfig.NUM_EPOCHS,
        per_device_train_batch_size = DeBERTaConfig.BATCH_SIZE_TRAIN,
        per_device_eval_batch_size  = DeBERTaConfig.BATCH_SIZE_EVAL,
        learning_rate               = DeBERTaConfig.LEARNING_RATE,
        warmup_steps                = DeBERTaConfig.WARMUP_STEPS,
        weight_decay                = DeBERTaConfig.WEIGHT_DECAY,
        logging_steps               = DeBERTaConfig.LOGGING_STEPS,
        # ── Evaluation ────────────────────────────────────────
        eval_strategy               = "steps",
        eval_steps                  = 200,
        # ── Checkpointing: DISABLED ───────────────────────────
        save_strategy               = "no",
        load_best_model_at_end      = False,
        # ── Precision: full fp32 ──────────────────────────────
        fp16                        = False,
        bf16                        = False,
        # ── Stability ────────────────────────────────────────
        max_grad_norm               = DeBERTaConfig.MAX_GRAD_NORM,
        dataloader_num_workers      = 2,
        report_to                   = "none",
    )

    # ── Trainer ───────────────────────────────────────────────
    # No EarlyStoppingCallback
    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_metrics,
    )

    # ── Train ─────────────────────────────────────────────────
    print("\nTraining ... (fp32, no checkpointing)")
    print("Expect loss to drop below 0.5 by step 400 if working.\n")
    trainer.train()
    print("\n✅ Training complete — using final in-memory weights")

    # ── Sanity check on validation set ───────────────────────
    val_out = trainer.evaluate()
    val_auc = val_out.get("eval_roc_auc", float("nan"))
    print(f"\n  Validation AUC : {val_auc:.4f}  "
          f"{'✅ looks good' if val_auc > 0.7 else '⚠️  still low — check logs'}")

    # ── Evaluate on all test splits ───────────────────────────
    print("\nRunning test-set predictions ...")
    results = {}

    for test_name, test_df in test_data_dict.items():
        test_ds = TextDetectionDataset(
            test_df["text"].tolist(),
            test_df["label_encoded"].tolist(),
            tokenizer, DeBERTaConfig.MAX_SEQ_LENGTH)

        preds  = trainer.predict(test_ds)
        probs  = torch.nn.functional.softmax(
                     torch.tensor(preds.predictions),
                     dim=-1).numpy()
        scores = probs[:, 1]
        y_true = test_df["label_encoded"].values
        y_pred = (scores > 0.5).astype(int)

        auc  = roc_auc_score(y_true, scores)
        acc  = accuracy_score(y_true, y_pred)
        bri  = brier_score_loss(y_true, scores)
        ll   = log_loss(y_true, scores)
        mh   = scores[y_true == 0].mean()
        ml   = scores[y_true == 1].mean()
        sep  = ml - mh

        results[test_name] = {
            "y_true":               y_true,
            "y_pred":               y_pred,
            "detectability_scores": scores,
            "roc_auc":              auc,
            "brier_score":          bri,
            "log_loss":             ll,
            "accuracy":             acc,
            "mean_human_score":     mh,
            "mean_llm_score":       ml,
            "score_separation":     sep,
        }
        print(f"  {test_name:20s}  AUC={auc:.4f}  Acc={acc:.4f}  "
              f"MeanH={mh:.3f}  MeanL={ml:.3f}  Sep={sep:.3f}")

    # ── Save final model & tokenizer ──────────────────────────
    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n✅ Model saved → {output_dir}")

    # ── Save npy score files for Notebook 6 ──────────────────
    for eval_name, d in results.items():
        np.save(
            f"deberta_scores_{eval_name}.npy",
            {"y_true":  d["y_true"],
             "y_score": d["detectability_scores"]}
        )

    del model, trainer
    torch.cuda.empty_cache()
    return results


# ── Cell 7: Train on HC3 ───────────────────────────────────────
deberta_hc3_results = train_deberta(
    train_data     = hc3_train_split,
    val_data       = hc3_val_split,
    test_data_dict = {
        "hc3_to_hc3":  hc3_test,
        "hc3_to_eli5": eli5_test,
    },
    output_dir = DeBERTaConfig.get_output_dir("hc3"),
    tag        = "HC3",
)

# ── Cell 8: Train on ELI5 ─────────────────────────────────────
deberta_eli5_results = train_deberta(
    train_data     = eli5_train_split,
    val_data       = eli5_val_split,
    test_data_dict = {
        "eli5_to_eli5": eli5_test,
        "eli5_to_hc3":  hc3_test,
    },
    output_dir = DeBERTaConfig.get_output_dir("eli5"),
    tag        = "ELI5",
)

# ── Cell 9: Consolidate ───────────────────────────────────────
deberta_results = {
    "DeBERTa-v3": {
        **deberta_hc3_results,
        **deberta_eli5_results,
    }
}

EVAL_ORDER = ["hc3_to_hc3", "hc3_to_eli5",
              "eli5_to_eli5", "eli5_to_hc3"]

# ── Cell 10: Score Distributions ──────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("DeBERTa-v3-base: Detectability Score Distributions",
             fontsize=15, fontweight="bold")

for j, eval_name in enumerate(EVAL_ORDER):
    d  = deberta_results["DeBERTa-v3"][eval_name]
    ax = axes[j]
    ax.hist(d["detectability_scores"][d["y_true"] == 0],
            bins=30, alpha=0.6, label="Human",
            color="#3498db", range=(0, 1))
    ax.hist(d["detectability_scores"][d["y_true"] == 1],
            bins=30, alpha=0.6, label="LLM",
            color="#e74c3c", range=(0, 1))
    ax.set_title(f"{eval_name}\nAUC={d['roc_auc']:.4f}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Detectability Score")
    ax.set_ylabel("Frequency")
    ax.axvline(0.5, color="k", linestyle="--", alpha=0.4)
    ax.legend(); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(f"{DeBERTaConfig.RESULTS_DIR}/deberta_detectability.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Cell 11: Calibration Curves ───────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("DeBERTa-v3-base: Calibration Curves",
             fontsize=15, fontweight="bold")

for j, eval_name in enumerate(EVAL_ORDER):
    d  = deberta_results["DeBERTa-v3"][eval_name]
    ax = axes[j]
    fop, mpv = calibration_curve(
        d["y_true"], d["detectability_scores"],
        n_bins=10, strategy="uniform")
    ax.plot(mpv, fop, "s-", label="DeBERTa-v3",
            linewidth=2.5, color="#2ecc71")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect")
    ax.set_title(eval_name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Mean Predicted")
    ax.set_ylabel("Fraction LLM")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DeBERTaConfig.RESULTS_DIR}/deberta_calibration.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Cell 12: ROC Curves ───────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("DeBERTa-v3-base: ROC Curves",
             fontsize=15, fontweight="bold")

for j, eval_name in enumerate(EVAL_ORDER):
    d  = deberta_results["DeBERTa-v3"][eval_name]
    ax = axes[j]
    fpr, tpr, _ = roc_curve(d["y_true"], d["detectability_scores"])
    ax.plot(fpr, tpr, linewidth=2.5, color="#9b59b6",
            label=f"AUC={d['roc_auc']:.4f}")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random")
    ax.set_title(eval_name, fontsize=11, fontweight="bold")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend(fontsize=9, loc="lower right"); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DeBERTaConfig.RESULTS_DIR}/deberta_roc.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Cell 13: Performance Summary ──────────────────────────────
rows = []
for eval_name in EVAL_ORDER:
    d = deberta_results["DeBERTa-v3"][eval_name]
    rows.append({
        "Detector":          "DeBERTa-v3-base",
        "Evaluation":        eval_name,
        "ROC-AUC":           d["roc_auc"],
        "Accuracy":          d["accuracy"],
        "Brier Score":       d["brier_score"],
        "Log Loss":          d["log_loss"],
        "Mean Human Score":  d["mean_human_score"],
        "Mean LLM Score":    d["mean_llm_score"],
        "Score Separation":  d["score_separation"],
    })

summary_df = pd.DataFrame(rows)

print("\n" + "=" * 70)
print("DEBERTA-v3 PERFORMANCE SUMMARY")
print("=" * 70)
print(summary_df.round(4).to_string(index=False))

print("\n── Sanity checks ──────────────────────────────────────")
all_ok = True
for _, row in summary_df.iterrows():
    if row["ROC-AUC"] < 0.7:
        print(f"  ⚠️  {row['Evaluation']:20s}  "
              f"AUC={row['ROC-AUC']:.4f}  — unexpectedly low")
        all_ok = False
    elif row["Score Separation"] < 0.3:
        print(f"  ⚠️  {row['Evaluation']:20s}  "
              f"Sep={row['Score Separation']:.4f}  — low separation")
        all_ok = False
    else:
        print(f"  ✅ {row['Evaluation']:20s}  "
              f"AUC={row['ROC-AUC']:.4f}  "
              f"Sep={row['Score Separation']:.4f}")

if all_ok:
    print("\n✅ All evaluations look healthy")

# ── Save ──────────────────────────────────────────────────────
summary_df.to_csv(
    f"{DeBERTaConfig.RESULTS_DIR}/deberta_performance_summary.csv",
    index=False)

with open(f"{DeBERTaConfig.RESULTS_DIR}/deberta_results.pkl", "wb") as f:
    pickle.dump(deberta_results, f)

print(f"\n🎯 DeBERTa-v3-base training complete.")